# 🚀 Modèles Avancés et Optimisation

## Objectif
Tester des modèles avancés pour améliorer les performances:
- Random Forest
- LightGBM
- Hyperparameter tuning
- Ensemble methods
- Fine-tuning de Transformers (bonus)

In [1]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Modules locaux
from src.data_loader import load_training_data, load_test_data, save_submission
from src.preprocessing import extract_full_text
from src.models import LogisticRegressionModel, RandomForestModel
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curve
from src.utils import load_config

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Imports réussis")

✓ Imports réussis


## 1. Chargement et Préparation

In [ ]:
# Charger config
config = load_config('../config/config.json')
print("✓ Configuration chargée")

# Charger données (utiliser load_jsonl pour avoir le DataFrame complet)
print("\n📥 Chargement des données...")
from src.data_loader import load_jsonl

df_train = load_jsonl('../data/train.jsonl')
df_test = load_jsonl('../data/kaggle_test.jsonl')

# Extraire texte
df_train['full_text'] = df_train.apply(extract_full_text, axis=1)
df_test['full_text'] = df_test.apply(extract_full_text, axis=1)

# Agréger par utilisateur
train_grouped = df_train.groupby('challenge_id').agg({
    'full_text': lambda x: ' '.join(x.astype(str)),
    'label': 'first'
}).reset_index()

test_grouped = df_test.groupby('challenge_id').agg({
    'full_text': lambda x: ' '.join(x.astype(str))
}).reset_index()

X_train = train_grouped['full_text']
y_train = train_grouped['label']
X_test = test_grouped['full_text']
test_ids = test_grouped['challenge_id']

print(f"✓ Train: {len(X_train)} utilisateurs")
print(f"✓ Test: {len(X_test)} utilisateurs")

✓ Configuration chargée

📥 Chargement des données...


AttributeError: 'tuple' object has no attribute 'apply'

## 2. Baseline: Logistic Regression

In [ ]:
print("📊 Baseline: Logistic Regression\n")

lr_model = LogisticRegressionModel(
    max_features=1000,
    C=1.0,
    random_state=42
)

# Validation croisée
cv_results = lr_model.cross_validate(X_train, y_train, cv=5)
print(f"Accuracy (CV): {cv_results['mean_score']:.4f} (+/- {cv_results['std_score']:.4f})")

# Entraîner sur tout le train
lr_model.fit(X_train, y_train)
print("✓ Modèle entraîné")

## 3. Random Forest

In [ ]:
print("🌲 Random Forest\n")

rf_model = RandomForestModel(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    max_features=1000,
    random_state=42,
    n_jobs=-1
)

# Validation croisée
cv_results_rf = rf_model.cross_validate(X_train, y_train, cv=5)
print(f"Accuracy (CV): {cv_results_rf['mean_score']:.4f} (+/- {cv_results_rf['std_score']:.4f})")

# Entraîner
rf_model.fit(X_train, y_train)
print("✓ Modèle entraîné")

## 4. LightGBM

In [ ]:
print("💡 LightGBM\n")

import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

# Pipeline
lgb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=1000,
        ngram_range=(1, 2),
        min_df=3
    )),
    ('clf', lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=-1,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(lgb_pipeline, X_train, y_train, cv=cv, scoring='accuracy')
print(f"Accuracy (CV): {scores.mean():.4f} (+/- {scores.std():.4f})")

# Entraîner
lgb_pipeline.fit(X_train, y_train)
print("✓ Modèle entraîné")

## 5. Comparaison des Modèles

In [ ]:
# Résumé des performances
results_summary = pd.DataFrame({
    'Modèle': ['Logistic Regression', 'Random Forest', 'LightGBM'],
    'Accuracy (CV)': [
        cv_results['mean_score'],
        cv_results_rf['mean_score'],
        scores.mean()
    ],
    'Std': [
        cv_results['std_score'],
        cv_results_rf['std_score'],
        scores.std()
    ]
})

print("\n📊 Comparaison des Performances:\n")
print(results_summary.to_string(index=False))

# Visualisation
plt.figure(figsize=(10, 6))
x = range(len(results_summary))
plt.bar(x, results_summary['Accuracy (CV)'], yerr=results_summary['Std'], 
        color=['skyblue', 'lightgreen', 'salmon'], capsize=5)
plt.xticks(x, results_summary['Modèle'])
plt.ylabel('Accuracy (CV)')
plt.title('Comparaison des Modèles', fontsize=14, fontweight='bold')
plt.ylim(0, 1)
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Baseline (50%)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning (Optionnel)

In [ ]:
# Exemple: Grid Search pour Logistic Regression
print("🔧 Hyperparameter Tuning (Logistic Regression)\n")

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Pipeline pour grid search
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

# Paramètres à tester
param_grid = {
    'tfidf__max_features': [500, 1000, 2000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.1, 1.0, 10.0]
}

# Grid Search (peut prendre du temps!)
# grid_search = GridSearchCV(lr_pipe, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
# grid_search.fit(X_train, y_train)
# print(f"\nMeilleurs paramètres: {grid_search.best_params_}")
# print(f"Meilleur score: {grid_search.best_score_:.4f}")

print("⚠️ Grid Search commenté (temps d'exécution)")
print("💡 Décommentez le code ci-dessus pour l'exécuter")

## 7. Prédictions sur Test Set

In [ ]:
# Sélectionner le meilleur modèle (basé sur CV)
best_model_name = results_summary.loc[results_summary['Accuracy (CV)'].idxmax(), 'Modèle']
print(f"🏆 Meilleur modèle: {best_model_name}\n")

# Utiliser le modèle correspondant
if best_model_name == 'Logistic Regression':
    best_model = lr_model
elif best_model_name == 'Random Forest':
    best_model = rf_model
else:
    best_model = lgb_pipeline

# Prédictions
print("🔮 Prédictions sur le test set...")
predictions = best_model.predict(X_test)

print(f"✓ {len(predictions)} prédictions générées")
print(f"\nDistribution:")
print(f"  - Observers (0): {(predictions == 0).sum()}")
print(f"  - Influencers (1): {(predictions == 1).sum()}")

In [ ]:
# Créer le fichier de soumission
submission_file = f'../submissions/{best_model_name.lower().replace(" ", "_")}_submission.csv'
save_submission(test_ids, predictions, submission_file)
print(f"✓ Soumission sauvegardée: {submission_file}")

## 8. Transformers (Bonus - Avancé)

In [ ]:
# Note: Cette section nécessite beaucoup de ressources (GPU recommandé)
# Elle est commentée par défaut

print("🤖 Fine-tuning de CamemBERT (commenté)")
print("\n💡 Pour utiliser CamemBERT:")
print("   1. Installez: pip install transformers torch")
print("   2. Utilisez un GPU (Google Colab gratuit)")
print("   3. Suivez les tutoriels HuggingFace")

# from transformers import CamembertTokenizer, CamembertForSequenceClassification
# from transformers import Trainer, TrainingArguments
# 
# model_name = "camembert-base"
# tokenizer = CamembertTokenizer.from_pretrained(model_name)
# model = CamembertForSequenceClassification.from_pretrained(model_name, num_labels=2)
# ...

print("\n⚠️ Code Transformers non exécuté (commenté)")

## 9. Conclusions

### Performance:
- Le meilleur modèle identifié
- Comparaison avec baseline
- Potentiel d'amélioration

### Prochaines étapes:
- Soumettre sur Kaggle
- Analyser les erreurs
- Feature engineering additionnel
- Ensemble methods
- Fine-tuning de transformers (si ressources disponibles)

In [ ]:
print("✅ Notebook terminé!")
print(f"\n📤 Soumettez {submission_file} sur Kaggle")
print("🎯 Bonne chance pour le challenge!")